# Redes Neuronales Convolucionales

Notebook mejorado: incluye EDA completo, arquitectura CNN con BatchNorm/Dropout,
entrenamiento con scheduler y early stopping, curvas de aprendizaje,
matriz de confusión y evaluación cualitativa.

## Introducción

Las **redes neuronales convolucionales** (CNNs) surgieron del estudio del córtex visual del cerebro y se han utilizado en reconocimiento de imágenes desde la década de los 80. Gracias al aumento de la potencia computacional y la disponibilidad de datos, las CNNs logran rendimiento sobrehumano en tareas visuales complejas: búsqueda de imágenes, coches autónomos, clasificación de vídeo, y más.

### El córtex visual

Las neuronas del córtex visual tienen un *campo receptivo local*: reaccionan solo a estímulos en una región limitada del campo visual. Las primeras capas detectan patrones simples (líneas, bordes), las siguientes detectan patrones más complejos (texturas, formas). Las CNNs replican esta jerarquía.

## 1. Imports y configuración del dispositivo

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import random
import os
import scipy.signal
from skimage import color, exposure

# sklearn: pip install scikit-learn seaborn
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Carga del dataset

In [ ]:
DATASET_DIR = r"c:\\Users\\lucia\\Desktop\\CNN dataset\\dataset_v2"

# ── Transforms ──────────────────────────────────────────────────────────
# Data augmentation SOLO en train
transform_train = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Val y test: solo resize + normalize (sin augmentation)
transform_val = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Transform sin normalize, para visualización
transform_vis = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
])

trainset     = ImageFolder(root=DATASET_DIR + "/train", transform=transform_train)
valset       = ImageFolder(root=DATASET_DIR + "/val",   transform=transform_val)
testset      = ImageFolder(root=DATASET_DIR + "/test",  transform=transform_val)
trainset_vis = ImageFolder(root=DATASET_DIR + "/train", transform=transform_vis)

classes = trainset.classes
print(f"Clases ({len(classes)}): {classes}")
print(f"Train: {len(trainset)} | Val: {len(valset)} | Test: {len(testset)}")

## 3. Análisis Exploratorio de Datos (EDA)

### 3.1 Distribución de clases
Detectamos posibles desbalances entre clases antes de entrenar.

In [ ]:
labels_train = [trainset[i][1] for i in range(len(trainset))]
counts = np.bincount(labels_train)

fig, ax = plt.subplots(figsize=(max(8, len(classes)*1.2), 4))
colors = plt.cm.tab10(np.linspace(0, 1, len(classes)))
bars = ax.bar(classes, counts, color=colors, edgecolor='white')
ax.set_title("Distribución de clases en TRAIN", fontsize=14, fontweight='bold')
ax.set_xlabel("Clase")
ax.set_ylabel("Cantidad de imágenes")
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01,
            str(cnt), ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=30, ha='right')
ax.set_ylim(0, max(counts) * 1.12)
plt.tight_layout()
plt.show()

print(f"Clase mayoritaria: {classes[np.argmax(counts)]} -> {max(counts)} imgs")
print(f"Clase minoritaria: {classes[np.argmin(counts)]} -> {min(counts)} imgs")
print(f"Ratio desbalance: {round(max(counts) / min(counts), 1)}:1")

### 3.2 Una muestra por clase

In [ ]:
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

def denormalize(tensor):
    """Convierte tensor normalizado -> imagen numpy visible."""
    img = tensor.permute(1, 2, 0).numpy()
    return np.clip(std * img + mean, 0, 1)

# Recopilar una imagen por clase
samples = {}
for img_t, label in trainset:
    if label not in samples:
        samples[label] = img_t
    if len(samples) == len(classes):
        break

cols = min(5, len(classes))
rows = (len(classes) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))
axes = np.array(axes).flatten()

for i, label in enumerate(sorted(samples.keys())):
    img = denormalize(samples[label])
    axes[i].imshow(img)
    axes[i].set_title(classes[label], fontsize=10, fontweight='bold')
    axes[i].axis("off")

for j in range(len(samples), len(axes)):
    axes[j].axis("off")

plt.suptitle("Una muestra por clase", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 3.3 Visualización del Data Augmentation
Mostramos la misma imagen con distintas transformaciones aleatorias para verificar que el augmentation es razonable.

In [ ]:
idx = random.randint(0, len(trainset_vis) - 1)
_, base_label = trainset_vis[idx]

fig, axes = plt.subplots(2, 6, figsize=(16, 6))

# Fila superior: imagen original (sin aug) varias veces
# Fila inferior: con augmentation
transform_noaug = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])
trainset_noaug = ImageFolder(root=DATASET_DIR + "/train", transform=transform_noaug)

img_orig, _ = trainset_noaug[idx]
img_orig_np = img_orig.permute(1, 2, 0).numpy()

for col in range(6):
    # Original
    axes[0][col].imshow(img_orig_np)
    axes[0][col].set_title("Original" if col == 0 else "", fontsize=9)
    axes[0][col].axis("off")
    # Augmentada
    img_aug, _ = trainset_vis[idx]
    axes[1][col].imshow(img_aug.permute(1, 2, 0).numpy())
    axes[1][col].set_title(f"Aug #{col+1}", fontsize=9)
    axes[1][col].axis("off")

axes[0][0].set_ylabel("Sin aug", fontsize=10)
axes[1][0].set_ylabel("Con aug", fontsize=10)
plt.suptitle(f"Data Augmentation — clase: {classes[base_label]}", fontsize=13)
plt.tight_layout()
plt.show()

## 4. La Capa Convolucional

Antes de construir la CNN completa, entendemos qué hace una convolución aplicando filtros manualmente. En la práctica, la red **aprende** estos filtros durante el entrenamiento.

### 4.1 Filtros manuales sobre una imagen real

In [ ]:
ix = random.randint(0, len(trainset) - 1)
img_t, label = trainset[ix]
img = denormalize(img_t)          # imagen RGB desnormalizada
img_gray = color.rgb2gray(img)    # escala de grises para filtros 2D

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Original
axes[0].imshow(img)
axes[0].set_title(f"Original\n({classes[label]})", fontsize=11)
axes[0].axis("off")

# --- Filtro Sobel horizontal (bordes H) ---
kernel_h = np.array([[ 1,  1,  1],
                     [ 0,  0,  0],
                     [-1, -1, -1]])
edges_h = scipy.signal.convolve2d(img_gray, kernel_h, "valid")
edges_h = exposure.equalize_adapthist(
    edges_h / (np.max(np.abs(edges_h)) + 1e-8), clip_limit=0.03)
axes[1].imshow(edges_h, cmap=plt.cm.gray)
axes[1].set_title("Sobel Horizontal\n(bordes horizontales)", fontsize=11)
axes[1].axis("off")

# --- Filtro Sobel vertical (bordes V) ---
kernel_v = np.array([[1, 0, -1],
                     [1, 0, -1],
                     [1, 0, -1]])
edges_v = scipy.signal.convolve2d(img_gray, kernel_v, "valid")  # usa img_gray (2D)
edges_v = exposure.equalize_adapthist(
    edges_v / (np.max(np.abs(edges_v)) + 1e-8), clip_limit=0.03)
axes[2].imshow(edges_v, cmap=plt.cm.gray)
axes[2].set_title("Sobel Vertical\n(bordes verticales)", fontsize=11)
axes[2].axis("off")

# --- Filtro Laplaciano (todos los bordes) ---
kernel_lap = np.array([[0, -1,  0],
                       [-1, 4, -1],
                       [0, -1,  0]])
edges_lap = scipy.signal.convolve2d(img_gray, kernel_lap, "valid")
edges_lap = exposure.equalize_adapthist(
    edges_lap / (np.max(np.abs(edges_lap)) + 1e-8), clip_limit=0.03)
axes[3].imshow(edges_lap, cmap=plt.cm.gray)
axes[3].set_title("Laplaciano\n(todos los bordes)", fontsize=11)
axes[3].axis("off")

plt.suptitle("Filtros manuales — la CNN aprenderá filtros como estos", fontsize=13)
plt.tight_layout()
plt.show()

### 4.2 Efecto del padding y stride en las dimensiones

La dimensión de salida de una convolución se calcula como:

$$o = \\left\\lfloor \\frac{n + 2p - m}{s} \\right\\rfloor + 1$$

donde $n$ = tamaño de entrada, $p$ = padding, $m$ = tamaño del filtro, $s$ = stride.

In [ ]:
img_t_demo, _ = trainset[0]
img_tensor_demo = img_t_demo.unsqueeze(0)  # (1, C, H, W)
print(f"Shape entrada:              {img_tensor_demo.shape}")

c1 = nn.Conv2d(3, 10, kernel_size=3, padding=0, stride=1)
c2 = nn.Conv2d(3, 10, kernel_size=3, padding=1, stride=1)
c3 = nn.Conv2d(3, 10, kernel_size=3, padding=0, stride=2)

with torch.no_grad():
    print(f"Sin padding, stride=1:       {c1(img_tensor_demo).shape}")
    print(f"Con padding=1, stride=1:     {c2(img_tensor_demo).shape}  <- mismo tamaño")
    print(f"Sin padding, stride=2:       {c3(img_tensor_demo).shape}")

## 5. Arquitectura de la CNN

Diseñamos una CNN con:
- **3 bloques convolucionales** (Conv → BatchNorm → ReLU → Conv → BatchNorm → ReLU → MaxPool → Dropout)
- **Clasificador** fully-connected con Dropout
- **BatchNorm** para estabilizar y acelerar el entrenamiento
- **Dropout** para regularización y evitar overfitting

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            # ── Bloque 1: 64x64 -> 32x32 ──────────────────────────────
            nn.Conv2d(3,  32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.1),

            # ── Bloque 2: 32x32 -> 16x16 ──────────────────────────────
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),

            # ── Bloque 3: 16x16 -> 8x8 ────────────────────────────────
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.3),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = CNN(num_classes=len(classes)).to(device)
print(model)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parámetros entrenables: {total:,}")

## 6. DataLoaders

In [ ]:
BATCH_SIZE  = 64
NUM_WORKERS = 0   # Windows: mantener en 0

train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device=="cuda"))
val_loader   = DataLoader(valset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device=="cuda"))
test_loader  = DataLoader(testset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device=="cuda"))

print(f"Batches de train : {len(train_loader)}")
print(f"Batches de val   : {len(val_loader)}")
print(f"Batches de test  : {len(test_loader)}")

## 7. Entrenamiento

Usamos:
- **Adam** con weight decay para regularización
- **ReduceLROnPlateau**: reduce el LR cuando el val_loss deja de mejorar
- **Early Stopping**: detiene el entrenamiento si no mejora después de N épocas
- Se guarda el **mejor modelo** según val_loss

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5)

NUM_EPOCHS              = 60
EARLY_STOPPING_PATIENCE = 12
BEST_MODEL_PATH         = r"c:\\Users\\lucia\\Desktop\\CNN dataset\\best_model.pth"

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * imgs.size(0)
        total_correct += (outputs.argmax(1) == labels).sum().item()
        total         += imgs.size(0)
    return total_loss / total, total_correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss    += loss.item() * imgs.size(0)
            total_correct += (outputs.argmax(1) == labels).sum().item()
            total         += imgs.size(0)
    return total_loss / total, total_correct / total

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss    = float("inf")
patience_counter = 0

header = f"{'Epoch':>6}  {'TrainLoss':>10}  {'TrainAcc':>9}  {'ValLoss':>9}  {'ValAcc':>9}  {'LR':>8}"
print(header)
print("-" * len(header))

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_acc   = eval_epoch(model, val_loader,   criterion, device)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    lr = optimizer.param_groups[0]["lr"]
    print(f"{epoch:>6}  {train_loss:>10.4f}  {train_acc:>8.2%}  {val_loss:>9.4f}  {val_acc:>8.2%}  {lr:>8.6f}")

    # ── Checkpoint del mejor modelo ──
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\n⚠ Early stopping activado en época {epoch}")
            break

print(f"\n✓ Mejor val_loss: {best_val_loss:.4f}  (modelo guardado en {BEST_MODEL_PATH})")

## 8. Curvas de aprendizaje

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)

# Loss
ax1.plot(epochs_range, history["train_loss"], label="Train", color="royalblue", lw=2)
ax1.plot(epochs_range, history["val_loss"],   label="Val",   color="tomato",    lw=2)
ax1.set_title("Pérdida (Loss)", fontsize=13)
ax1.set_xlabel("Época")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs_range, [a*100 for a in history["train_acc"]], label="Train", color="royalblue", lw=2)
ax2.plot(epochs_range, [a*100 for a in history["val_acc"]],   label="Val",   color="tomato",    lw=2)
ax2.set_title("Precisión (Accuracy)", fontsize=13)
ax2.set_xlabel("Época")
ax2.set_ylabel("Accuracy (%)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("Curvas de entrenamiento", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Evaluación en el conjunto de Test

Cargamos el **mejor modelo** guardado y lo evaluamos en datos que nunca vio durante el entrenamiento.

In [ ]:
# Cargar mejor checkpoint
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
test_loss, test_acc = eval_epoch(model, test_loader, criterion, device)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.2%}")

### 9.1 Matriz de confusión

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs   = imgs.to(device)
        preds  = model(imgs).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

fig_size = max(8, len(classes))
fig, ax = plt.subplots(figsize=(fig_size, fig_size - 1))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes, ax=ax,
            linewidths=0.5, linecolor='gray')
ax.set_title("Matriz de Confusión — Test Set", fontsize=14, fontweight='bold')
ax.set_xlabel("Predicción", fontsize=12)
ax.set_ylabel("Real", fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### 9.2 Reporte de clasificación y precisión por clase

In [ ]:
print("Reporte de clasificación en TEST:\n")
print(classification_report(all_labels, all_preds, target_names=classes, digits=3))

# Precisión por clase
class_correct = cm.diagonal()
class_total   = cm.sum(axis=1)
print("\nPrecisión por clase:")
for i, cls in enumerate(classes):
    pct = 100 * class_correct[i] / class_total[i] if class_total[i] > 0 else 0
    bar = '#' * int(pct // 5)
    print(f"  {cls:20s}: {class_correct[i]:4d}/{class_total[i]:4d} = {pct:5.1f}%  {bar}")

### 9.3 Ejemplos cualitativos de predicciones

In [ ]:
model.eval()
n_samples = 15
sample_indices = random.sample(range(len(testset)), n_samples)

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
axes = axes.flatten()

with torch.no_grad():
    for ax, idx in zip(axes, sample_indices):
        img_t, true_label = testset[idx]
        output     = model(img_t.unsqueeze(0).to(device))
        probs      = torch.softmax(output, dim=1)[0]
        pred_label = probs.argmax().item()
        confidence = probs.max().item()

        img = denormalize(img_t)
        ax.imshow(img)
        ok    = pred_label == true_label
        color = "green" if ok else "red"
        icon  = "✓" if ok else "✗"
        ax.set_title(
            f"{icon} Real: {classes[true_label]}\n"
            f"Pred: {classes[pred_label]} ({confidence:.0%})",
            fontsize=8, color=color
        )
        ax.axis("off")

plt.suptitle("Predicciones en Test  (verde = correcto, rojo = incorrecto)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()